# Patterns 7 & 8: Loop, Review-Critique, and Iterative Refinement

> **Google Cloud References**:
> - [Loop Pattern](https://docs.cloud.google.com/architecture/choose-design-pattern-agentic-ai-system#loop-pattern)
> - [Review and Critique Pattern](https://docs.cloud.google.com/architecture/choose-design-pattern-agentic-ai-system#review-critique-pattern)
> - [Iterative Refinement Pattern](https://docs.cloud.google.com/architecture/choose-design-pattern-agentic-ai-system#iterative-refinement-pattern)

All three patterns share a **looping mechanism** — they differ in what drives the loop.

## Loop Pattern
- Repeatedly executes subagents until an **exit condition** is met
- Exit: max iterations, custom state, time limit
- No AI model for orchestration — predefined logic drives the loop

## Review and Critique Pattern (Generator-Critic)
- **Generator** creates output → **Critic** evaluates it → Loop until approved
- Exit: critic approves OR max iterations reached
- Use for: code review, content quality, safety checks

## Iterative Refinement Pattern
- Output is **progressively improved** over multiple cycles
- Each iteration refines based on the previous output
- Use for: complex writing, code generation, planning

```
[Generator] ──────► [Critic]
     ▲                  │
     │    (loop)        │ (approved?)
     └──── Feedback ◄──┘
                        │ (yes)
                        ▼
                  Final Output
```

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
for var in ['OPENAI_API_KEY', 'OPENAI_BASE_URL', 'OPENAI_API_BASE']:
    if os.getenv(var):
        del os.environ[var]

os.environ["OPENAI_API_KEY"] = os.getenv("OPEN_ROUTER_KEY")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

### Part A: Review and Critique Pattern — Code Generation + Security Review

> **Agno 2.x note**: The old `Workflow` base-class pattern (`class MyWorkflow(Workflow): def run(...) -> RunResponse`)
> was removed. Use plain Python classes instead — same loop logic, no framework dependency.
> `RunResponse` is also gone; return plain strings or use `result.content` from `agent.run()`.

In [ ]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from pydantic import BaseModel, Field
from typing import Optional

model = OpenAIChat(id="google/gemini-2.5-flash")

class CodeReview(BaseModel):
    approved: bool = Field(..., description="True if code passes review, False if revisions needed")
    security_score: int = Field(..., ge=0, le=10, description="Security score 0-10")
    issues: list[str] = Field(default_factory=list, description="List of issues found")
    feedback: str = Field(..., description="Detailed feedback for the generator")

def run_structured(agent: Agent, prompt: str, retries: int = 2):
    """gemini-2.5-flash occasionally returns truncated/malformed JSON, which
    agno can't parse into the output_schema — it falls back to a raw string.
    Retry a couple of times before giving up."""
    result = None
    for attempt in range(retries + 1):
        result = agent.run(prompt)
        if not isinstance(result.content, str):
            return result
        print(f"   ⚠️  {agent.name}: malformed structured output, retrying ({attempt + 1}/{retries})...")
    raise RuntimeError(
        f"Agent '{agent.name}' failed to return structured output after {retries + 1} attempts. "
        f"Last raw response: {result.content!r}"
    )

class ReviewCritiqueWorkflow:
    """
    Review and Critique Pattern: Generator → Critic → Loop until approved.
    Exit conditions: Critic approves OR max_iterations reached.
    """

    MAX_ITERATIONS = 3

    def __init__(self):
        # Generator: produces code
        self.code_generator = Agent(
            name="Code Generator",
            model=model,
            instructions=[
                "Generate clean, production-ready Python code.",
                "When given feedback, fix ALL issues mentioned by the security reviewer.",
                "Always include: input validation, error handling, type hints, docstrings.",
                "Format code in a ```python code block.",
            ],
            markdown=True,
        )

        # Critic: reviews for security
        self.security_reviewer = Agent(
            name="Security Reviewer",
            model=model,
            output_schema=CodeReview,  # Agno 2.x: was response_model= in older versions
            structured_outputs=True,
            instructions=[
                "You are a senior security engineer reviewing code.",
                "Check for: SQL injection, XSS, path traversal, hardcoded secrets, insecure defaults.",
                "Check for: missing input validation, unhandled exceptions, resource leaks.",
                "Only approve (approved=True) if security_score >= 8 and there are no critical issues.",
                "Be strict — a security score of 7 or below means send back for revision.",
            ],
        )

    def run(self, task: str) -> str:
        iteration = 0
        current_code = None
        feedback = None

        while iteration < self.MAX_ITERATIONS:
            iteration += 1
            print(f"\n{'─'*50}")
            print(f"🔄 Iteration {iteration}/{self.MAX_ITERATIONS}")

            # GENERATE
            prompt = task if iteration == 1 else (
                f"Revise this code based on security feedback:\n"
                f"Original task: {task}\n"
                f"Previous code:\n{current_code}\n"
                f"Security feedback to fix:\n{feedback}"
            )
            print("⚙️  Generator creating code...")
            gen_response = self.code_generator.run(prompt)
            current_code = gen_response.content

            # CRITIQUE
            print("🔍 Security reviewer evaluating...")
            review_response = run_structured(
                self.security_reviewer,
                f"Review this code for security:\n{current_code}"
            )
            review: CodeReview = review_response.content

            print(f"   Score: {review.security_score}/10 | Approved: {review.approved}")
            if review.issues:
                print(f"   Issues: {len(review.issues)} found")

            # EXIT CONDITION
            if review.approved:
                print(f"\n✅ APPROVED after {iteration} iteration(s)!")
                return f"## Approved Code (Score: {review.security_score}/10, Iterations: {iteration})\n\n{current_code}"

            feedback = review.feedback

        print(f"\n⚠️  Max iterations ({self.MAX_ITERATIONS}) reached — returning best version.")
        return f"## Code (Max iterations reached)\n\n{current_code}\n\n**Final feedback:** {feedback}"


review_workflow = ReviewCritiqueWorkflow()
result = review_workflow.run(
    "Write a Python function that accepts a username and password, "
    "queries a SQLite database to verify credentials, and returns a JWT token. "
    "Include proper security measures."
)
print("\n" + "="*65)
print("FINAL OUTPUT")
print("="*65)
print(result)


──────────────────────────────────────────────────
🔄 Iteration 1/3
⚙️  Generator creating code...


🔍 Security reviewer evaluating...


   Score: 3/10 | Approved: False
   Issues: 6 found

──────────────────────────────────────────────────
🔄 Iteration 2/3
⚙️  Generator creating code...


🔍 Security reviewer evaluating...


   Score: 9/10 | Approved: True

✅ APPROVED after 2 iteration(s)!

FINAL OUTPUT
## Approved Code (Score: 9/10, Iterations: 2)

```python
import jwt
import sqlite3
import os
import datetime
import bcrypt
from typing import Optional, Dict, Any

def authenticate_user(username: str, password: str, db_path: str, jwt_secret: str, jwt_algorithm: str = "HS256", token_expiration_minutes: int = 30) -> Optional[str]:
    """
    Authenticates a user against a SQLite database and issues a JWT token upon successful verification.
    Passwords must be securely hashed using bcrypt in the database.

    Args:
        username (str): The username to authenticate.
        password (str): The plain-text password to verify.
        db_path (str): The path to the SQLite database file.
        jwt_secret (str): The secret key used to sign the JWT token.
        jwt_algorithm (str): The algorithm to use for JWT signing (default: "HS256").
                             Common choices include "HS256", "HS384", 

### Part B: Iterative Refinement — Blog Post Writing

In [ ]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from pydantic import BaseModel, Field

model = OpenAIChat(id="google/gemini-2.5-flash")

class ContentQuality(BaseModel):
    quality_score: int = Field(..., ge=0, le=10)
    meets_standard: bool = Field(..., description="True if score >= 8")
    clarity: str = Field(..., description="Feedback on clarity")
    engagement: str = Field(..., description="Feedback on engagement")
    accuracy: str = Field(..., description="Feedback on technical accuracy")
    specific_improvements: list[str] = Field(..., description="Concrete improvements to make")

def run_structured(agent: Agent, prompt: str, retries: int = 2):
    """gemini-2.5-flash occasionally returns truncated/malformed JSON, which
    agno can't parse into the output_schema — it falls back to a raw string.
    Retry a couple of times before giving up."""
    result = None
    for attempt in range(retries + 1):
        result = agent.run(prompt)
        if not isinstance(result.content, str):
            return result
        print(f"   ⚠️  {agent.name}: malformed structured output, retrying ({attempt + 1}/{retries})...")
    raise RuntimeError(
        f"Agent '{agent.name}' failed to return structured output after {retries + 1} attempts. "
        f"Last raw response: {result.content!r}"
    )

class IterativeRefinementWorkflow:
    """
    Iterative Refinement: Write → Critique → Improve → Repeat.
    Each iteration improves the same piece until it meets the quality standard.
    """

    MAX_ITERATIONS = 3
    QUALITY_THRESHOLD = 8

    def __init__(self):
        self.writer = Agent(
            name="Technical Writer",
            model=model,
            instructions=[
                "Write clear, engaging technical blog posts.",
                "When revising, specifically address each point of feedback.",
                "Use concrete examples, analogies, and code snippets where helpful.",
                "Target audience: experienced software engineers moving into AI.",
            ],
            markdown=True,
        )

        self.quality_reviewer = Agent(
            name="Content Quality Reviewer",
            model=model,
            output_schema=ContentQuality,  # Agno 2.x: was response_model= in older versions
            structured_outputs=True,
            instructions=[
                "Evaluate blog posts on: technical accuracy, clarity, and reader engagement.",
                "Score out of 10. meets_standard=True only if score >= 8.",
                "Provide specific, actionable improvements — not vague feedback.",
                "Be constructive but honest.",
            ],
        )

    def run(self, topic: str) -> str:
        iteration = 0
        current_draft = None
        all_feedback = []
        review = None

        while iteration < self.MAX_ITERATIONS:
            iteration += 1
            print(f"\n{'─'*50}")
            print(f"✍️  Draft {iteration}/{self.MAX_ITERATIONS}")

            # WRITE/REFINE
            if iteration == 1:
                prompt = f"Write a 400-word technical blog post about: {topic}"
            else:
                feedback_summary = "\n".join([f"- {fb}" for fb in all_feedback[-1]])
                prompt = (
                    f"Revise this blog post. Address all feedback:\n\n"
                    f"Previous draft:\n{current_draft}\n\n"
                    f"Feedback to address:\n{feedback_summary}"
                )

            write_response = self.writer.run(prompt)
            current_draft = write_response.content
            print(f"   Draft length: {len(current_draft)} chars")

            # REVIEW
            review_response = run_structured(
                self.quality_reviewer,
                f"Review this blog post about '{topic}':\n{current_draft}"
            )
            review: ContentQuality = review_response.content
            all_feedback.append(review.specific_improvements)

            print(f"   Quality score: {review.quality_score}/10 | Meets standard: {review.meets_standard}")
            print(f"   Improvements needed: {len(review.specific_improvements)}")

            # EXIT CONDITION: quality threshold met
            if review.meets_standard:
                print(f"\n✅ Quality standard met (score={review.quality_score}) after {iteration} draft(s)!")
                return f"# Final Blog Post (Score: {review.quality_score}/10)\n\n{current_draft}"

        score = review.quality_score if review else "?"
        print(f"\n⚠️  Max iterations reached. Returning best draft (score: {score}/10).")
        return f"# Blog Post (Best effort)\n\n{current_draft}"


refinement_workflow = IterativeRefinementWorkflow()
result = refinement_workflow.run(
    "The ReAct pattern in agentic AI: how Reason-Act-Observe loops make agents smarter"
)
print("\n" + "="*65)
print(result[:2000] + "..." if len(result) > 2000 else result)


──────────────────────────────────────────────────
✍️  Draft 1/3


   Draft length: 102 chars


   Quality score: 7/10 | Meets standard: False
   Improvements needed: 5

──────────────────────────────────────────────────
✍️  Draft 2/3


   Draft length: 11015 chars


   Quality score: 9/10 | Meets standard: True
   Improvements needed: 4

✅ Quality standard met (score=9) after 2 draft(s)!

# Final Blog Post (Score: 9/10)

# Beyond Simple Scripts: Taming Complexity with the ReAct Pattern in AI Agents

The field of AI agents is evolving at a breathtaking pace. We're moving beyond simple, pre-programmed task execution into a realm where agents exhibit increasingly complex, human-like reasoning and interaction. But how do these agents achieve such sophisticated behavior? One of the most influential patterns emerging in this space is **ReAct: Reason-Act-Observe**.

This pattern empowers Large Language Models (LLMs) to go beyond one-shot responses, enabling them to deliberate, take actions, and course-correct based on feedback from the environment. For experienced software engineers, understanding ReAct is crucial for building the next generation of intelligent systems.

## The Problem ReAct Solves

Traditional LLM applications often act as sophisticated

### Pattern Comparison

| Pattern | Who loops? | Exit condition | Primary driver |
|---------|-----------|---------------|----------------|
| Loop | Loop agent | Custom state or max iterations | Predefined logic |
| Review & Critique | Generator + Critic | Critic approval | Quality gate |
| Iterative Refinement | Single/multi agent | Quality threshold or max iters | Progressive improvement |

**Key implementation rule**: Always define a `MAX_ITERATIONS` guard — infinite loops are the primary failure mode.